# Complainify AI — 10 · Sentiment & Priority Models

Companion notebook to the main *Complainify_AI_Study.ipynb*.

---
**Labeling convention:** `negative` is reserved for STRONG complaint language (stale/rotten/broken/stolen/...). Mild problem-reports ("water not getting on time, Wi-Fi issue") are labeled `neutral` - urgency is the priority model's job, not sentiment.

## 10: Sentiment & Priority Models — Evaluation

Two from-scratch Multinomial Naive Bayes models (Laplace smoothing, log-space, bigrams — same implementation as the category classifier). The **baseline** is the previous rule-based pipeline (from-scratch pattern sentiment + keyword priority) evaluated on the exact same 80/20 split.

In [1]:
import os, sys, json, csv, math, random
from collections import Counter, defaultdict
BASE = r'E:\Project-VI\workspace\ComplaintMgmtSystem'
sys.path.insert(0, os.path.join(BASE, 'ml'))
random.seed(42)
from classifier import MultinomialNB
rows = list(csv.DictReader(open(os.path.join(BASE, 'data', 'sentiment_dataset.csv'), encoding='utf-8')))
random.shuffle(rows)
split = int(len(rows) * 0.8)
train, test = rows[:split], rows[split:]
print('train:', len(train), '| test:', len(test))
SENT = {'positive': 0, 'neutral': 1, 'negative': 2}
PRI = {'low': 0, 'medium': 1, 'high': 2}
train_t = [r['complaint_text'] for r in train]
test_t = [r['complaint_text'] for r in test]
s_model = MultinomialNB(); s_model.fit(train_t, [SENT[r['sentiment_label']] for r in train])
p_model = MultinomialNB(); p_model.fit(train_t, [PRI[r['priority'].lower()] for r in train])
SENT_D = {v: k for k, v in SENT.items()}; PRI_D = {v: k for k, v in PRI.items()}
print('vocab sizes (sentiment, priority):', s_model.vocab_size, p_model.vocab_size)

train: 9600 | test: 2400


vocab sizes (sentiment, priority): 3154 3154


In [2]:
import os, sys, json, csv, math, random
from collections import Counter, defaultdict
BASE = r'E:\Project-VI\workspace\ComplaintMgmtSystem'
sys.path.insert(0, os.path.join(BASE, 'ml'))
random.seed(42)
cls_s = ['positive', 'neutral', 'negative']
cls_p = ['low', 'medium', 'high']
def confusion(model, texts, true_labels, decoder, classes):
    cm = {t: {p: 0 for p in classes} for t in classes}
    for t, y in zip(texts, true_labels):
        pred, _ = model.predict_with_proba(t)
        cm[decoder[y]][decoder[pred]] += 1
    return cm
def per_class(cm, classes):
    out = []
    for c in classes:
        tp = cm[c][c]; fp = sum(cm[k][c] for k in classes if k != c)
        fn = sum(cm[c][k] for k in classes if k != c)
        p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
        out.append((c, p, r, 2 * p * r / max(p + r, 1e-9), tp + fn))
    return out
cm_s = confusion(s_model, test_t, [SENT[r['sentiment_label']] for r in test], SENT_D, cls_s)
cm_p = confusion(p_model, test_t, [PRI[r['priority'].lower()] for r in test], PRI_D, cls_p)
print('SENTIMENT confusion (rows=true, cols=pred):')
for k, v in cm_s.items(): print('  ', k, v)
print('PRIORITY confusion (rows=true, cols=pred):')
for k, v in cm_p.items(): print('  ', k, v)
print()
print('SENTIMENT per-class (test):')
for (c, p, r, f1, n) in per_class(cm_s, cls_s):
    print('  {:<8} P {:.3f} R {:.3f} F1 {:.3f}  n={}'.format(c, p, r, f1, n))
print('PRIORITY per-class (test):')
for (c, p, r, f1, n) in per_class(cm_p, cls_p):
    print('  {:<8} P {:.3f} R {:.3f} F1 {:.3f}  n={}'.format(c, p, r, f1, n))

SENTIMENT confusion (rows=true, cols=pred):
   positive {'positive': 819, 'neutral': 0, 'negative': 2}
   neutral {'positive': 3, 'neutral': 1060, 'negative': 112}
   negative {'positive': 0, 'neutral': 0, 'negative': 404}
PRIORITY confusion (rows=true, cols=pred):
   low {'low': 1211, 'medium': 3, 'high': 2}
   medium {'low': 11, 'medium': 744, 'high': 3}
   high {'low': 0, 'medium': 0, 'high': 426}

SENTIMENT per-class (test):
  positive P 0.996 R 0.998 F1 0.997  n=821
  neutral  P 1.000 R 0.902 F1 0.949  n=1175
  negative P 0.780 R 1.000 F1 0.876  n=404
PRIORITY per-class (test):
  low      P 0.991 R 0.996 F1 0.993  n=1216
  medium   P 0.996 R 0.982 F1 0.989  n=758
  high     P 0.988 R 1.000 F1 0.994  n=426


### Rule-based baseline on the same split

In [3]:
from sentiment import analyze_sentiment_rules as rule_sent
from priority import compute_priority_rules as rule_prio
def rule_sent_enc(t): return SENT[rule_sent(t)['label'].lower()]
def rule_prio_enc(t):
    s = rule_sent(t)
    p, sc, reason = rule_prio(t, s['label'], s['score'])
    return PRI[p.lower()]
def eval_run(pred_fn, true_labels):
    classes = sorted(set(true_labels))
    preds = [pred_fn(t) for t in test_t]
    acc = sum(1 for p, y in zip(preds, true_labels) if p == y) / len(test_t)
    per = {c: dict(tp=0, fp=0, fn=0) for c in classes}
    for p, y in zip(preds, true_labels):
        if p == y: per[y]['tp'] += 1
        else: per[p]['fp'] += 1; per[y]['fn'] += 1
    f1s = [2 * v['tp'] / max(2 * v['tp'] + v['fp'] + v['fn'], 1) for v in per.values()]
    return acc, sum(f1s) / len(f1s)
acc_s_r, f1_s_r = eval_run(rule_sent_enc, [SENT[r['sentiment_label']] for r in test])
acc_p_r, f1_p_r = eval_run(rule_prio_enc, [PRI[r['priority'].lower()] for r in test])
print('rule-based sentiment  acc {:.2f}%  macro-F1 {:.3f}'.format(acc_s_r * 100, f1_s_r))
print('rule-based priority   acc {:.2f}%  macro-F1 {:.3f}'.format(acc_p_r * 100, f1_p_r))


rule-based sentiment  acc 72.08%  macro-F1 0.705


rule-based priority   acc 59.25%  macro-F1 0.372


In [4]:
import datetime, json
log_p = os.path.join(BASE, 'ml', 'sentiment_training_log.json')
if os.path.isfile(log_p):
    log = json.load(open(log_p))
    print('training log (single source of truth for report tables):')
    print('  sentiment acc {}%  macro-F1 {}'.format(log['sentiment']['accuracy'], log['sentiment']['macro_f1']))
    print('  priority  acc {}%  macro-F1 {}'.format(log['priority']['accuracy'], log['priority']['macro_f1']))
else:
    print('no training log yet — run ml/train_sentiment_priority.py')

training log (single source of truth for report tables):
  sentiment acc 95.12%  macro-F1 0.9406
  priority  acc 99.21%  macro-F1 0.9921
